# Parte 3 — Redes neurais, embeddings, Transformers e LLMs

In [ ]:
%pip -q install datasets scikit-learn matplotlib

In [1]:
# Libs

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1
Dispositivo: cuda
GPU: AMD Radeon RX 9070


### Redes neurais artificiais, neurônios e perceptron
Uma **rede neural artificial** é um modelo computacional inspirado, de forma simplificada, pelo funcionamento de redes de neurônios biológicos. Ela é formada por undiades conectadas que recebem entradas, *combinam essas entradas usando pesos*, e produzem *saídas*. Uma **rede neural artificial** é um modelo computacional inspirado, de forma simplificada, no funcionamento de redes de neurônios biológicos. Ela é formada por unidades conectadas que recebem entradas, combinam essas entradas usando pesos e produzem saídas. Durante o treinamento, os pesos são ajustados para que o modelo faça previsões cada vez mais adequadas aos exemplos observados.

Se isso soa familiar, é por que é mesmo. Em uma perspectiva isoladam *um neurônio é quase igual a uma modelo linear* como os que demonstramos anteriormente, que recebe dados e devolva dados transformados linearmente. Porém, com algumas distinções importantes. A começar que, assim como neurônios cerebrais não trabalham sozinhos, *redes neurais são compostas de vários neurônios encadeados* de modo que trabalhem juntos para se chegar em um resultado. Outra distinção importante, é que os neurônios comumente possuem uma *função de ativação* que introduzem não-linearidade, como explicaremos em breve.


A unidade básica de uma rede neural é o **neurônio artificial**. Para entradas $x_1, x_2, \ldots, x_n$, pesos $w_1, w_2, \ldots, w_n$ e um viés $b$, o neurônio calcula primeiro uma combinação ponderada:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

Em seguida, uma função de ativação pode transformar $z$ na saída do neurônio:

$$
a = f(z)
$$

E esta é a estrutura básica de um neurônico artificial. Observe a semelhança com os modelos que já vimos até então.

In [3]:
### == CÓDIGO ==

### O Perceptron

O **perceptron**, proposto por Frank Rosenblatt no final da década de 1950, foi um dos primeiros modelos de neurônio artificial. Em sua concepção original, o perceptron se assemelhava muito ao modelo de *Regressão Logística*, com a diferença de que usamos uma função de ativação diferente, chamada de *degrau*, enquanto a logística utiliza a sigmoide. Existem mais de um tipo de função degrau, mas a mais conhecida é a *heaviside*:

$$
a = \begin{cases} 1, & \text{se } z \geq 0 \\ 0, & \text{se } z < 0 \end{cases}
$$s.

Note que, usar *um único perceptron* e utilizar uma função de ativação como a *sigmoide* como função de ativação, fará o perceptron agir exatamente como uma regressão logística.

### Por que precisamos de uma função de ativação?

A Álgebra Linear nos fornece as ferramentas necessárias para entender profundamente essa questão, mas de maneira intuitiva, podemos entender que as *transformações lineares* (como o que o neurônio artificial produz) possuem uma importante propriedade: ao serem combinadas, o todo *permanece linear*, isto é, transformações lineares em sequência podem ser representadas como uma única grnade transformação linear. 

Isso é ruim para o modelo, pois 'obriga' o modelo a apenas aprender padrões lineares. Como exemplo didático, imagina que seu modelo quer aprender a classificar imagens: um modelo 100% linear teria, nessas condições, extrema dificuldade para representar curvas suaves, por exemplo.

Assim, aplicamos da *função de ativação* para introduzir não-linearidade entre as *camadas* da rede neural, permitindo que o modelo aprenda padrões mais complexos do que simples relações lineares nos dados.


Algumas funções de ativação comuns são:

- **ReLU:**

$$
\operatorname{ReLU}(z) = \max(0, z)
$$

- **Sigmoide:**

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Entre outras.

In [ ]:
## == CÓDIGO ==

### Camadas e rede de neurônios
Até agora, vimos o que é um neurônio artificial. Porém, um único neurônio artificial isolado não é muito melhor que um modelo de regressão simples. O verdadeiro poder dos neurônios se dá, principalmente, quando eles se unem para formar uma *rede neural*. De forma simplificada, nós temos diferentes tipos de camadas com funções diferentes dentro do modelo de Rede Neural:

##### Características e vetor de entrada

Uma **característica** (*feature*) é uma informação usada pelo modelo para fazer uma previsão. Por exemplo, para prever se um estudante será aprovado, podemos usar:

- horas de estudo;
- número de exercícios resolvidos.

Cada exemplo é representado por um conjunto de características. Neste caso, temos duas características, que podem ser organizadas no vetor:

$$
x = [x_1, x_2]
$$

O vetor $x$ representa uma observação. Os valores $x_1$ e $x_2$ são as características daquele exemplo.

#### Camada de entrada

A **camada de entrada** é a porta de entrada dos dados na rede. Ela recebe o vetor de características e encaminha seus valores para a primeira camada de neurônios.

Se cada exemplo possui duas características, a entrada tem dimensão 2. A camada de entrada não precisa realizar uma transformação complexa: sua função principal é organizar os dados que serão processados pela rede.

#### Camada escondida e seus neurônios

A **camada escondida** é uma camada intermediária entre a entrada e a saída. Ela é formada por neurônios que combinam as características usando pesos e vieses próprios.

Cada neurônio calcula uma combinação ponderada das entradas:

$$
z_j = w_{j1}x_1 + w_{j2}x_2 + b_j
$$

Depois, aplica uma função de ativação:

$$
h_j = f(z_j)
$$

O índice $j$ identifica o neurônio. Como cada neurônio possui pesos diferentes, cada um pode aprender uma combinação diferente das mesmas características. Os valores produzidos nessa etapa são representações intermediárias, por isso não são diretamente os rótulos ou as previsões finais.

#### Camada de saída

A **camada de saída** recebe os valores produzidos pela camada escondida e transforma essa representação no resultado da tarefa.

Em uma classificação binária, podemos ter um único neurônio de saída. Ele produz um valor chamado **logit**:

$$
z_{saída} = W_2h + b_2
$$

O logit ainda não é uma probabilidade. Quando necessário, usamos a sigmoide para convertê-lo em um valor entre 0 e 1.

In [2]:
# == CÓDIGO ==

O ato de inicializar os valores para todos os pesos e vieses de todos os neurônios, e avançar pela primeira iteração completa do modelo até o final (mesmo com os pesos geralmente iguais a zero), é chamado de *feedforward* ou *passada para frente*.

In [ ]:
# Implementar a forward propagation passo a passo.
# Definir a entrada, os pesos, os vieses e as funções de ativação.
# Calcular a saída da camada escondida e a previsão final.


### Função de Custo, Backpropagation e Treinamento -- como modelos 'Aprendem'

A forward propagation produz uma previsão, mas ainda precisamos medir se ela está boa. A **função de custo** (ou função de perda) compara a previsão $\hat{y}$ com o valor correto $y$ e transforma essa diferença em um número. Quanto menor o custo, mais próximas estão as previsões dos rótulos esperados.

Para uma classificação binária, uma escolha comum é a **entropia cruzada binária** (*Binary Cross-Entropy*):

$$
J = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]
$$

A fórmula penaliza previsões confiantes e erradas. Por exemplo, quando $y=1$, uma probabilidade próxima de 1 gera custo baixo; uma probabilidade próxima de 0 gera custo alto.

Depois de calcular o custo, usamos o **backpropagation** (retropropagação). Ele aplica a regra da cadeia para descobrir como o custo varia em relação a cada peso e viés da rede:

$$
\frac{\partial J}{\partial W}, \qquad \frac{\partial J}{\partial b}
$$

Essas derivadas são chamadas de **gradientes**. Elas indicam a direção em que cada parâmetro aumenta o custo. O backpropagation não atualiza os parâmetros; ele apenas calcula esses gradientes.

O treinamento usa os gradientes para atualizar os parâmetros com o gradiente descendente:

$$
\theta \leftarrow \theta - \eta\nabla_{\theta}J
$$

Aqui, $\theta$ representa todos os pesos e vieses, $\eta$ é a taxa de aprendizado e $\nabla_{\theta}J$ é o gradiente do custo em relação aos parâmetros.

O ciclo completo de treinamento é:

1. realizar a forward propagation;
2. calcular a função de custo;
3. zerar gradientes antigos;
4. executar o backpropagation;
5. atualizar os parâmetros;
6. repetir por várias épocas.

Uma **época** corresponde a uma passagem pelo conjunto de treinamento. Em conjuntos grandes, os exemplos são divididos em lotes (*batches*), e os parâmetros são atualizados após cada lote.

In [ ]:
# Implementar a função de custo.
# Calcular os gradientes com backpropagation.
# Criar o ciclo de treinamento e atualizar os parâmetros.


### Exemplo de Rede Neural simples

Agora vamos reunir os conceitos anteriores em uma rede neural pequena para um problema de classificação binária. Imagine que queremos prever se um estudante será aprovado ou reprovado usando duas características: horas de estudo e quantidade de exercícios resolvidos.

A rede terá:

- duas entradas, uma para cada característica;
- uma camada escondida com quatro neurônios;
- uma função de ativação ReLU na camada escondida;
- um neurônio de saída, que produzirá um logit;
- uma sigmoide na etapa de previsão para obter uma probabilidade.

A arquitetura pode ser representada assim:

$$
X \rightarrow Linear(2, 4) \rightarrow ReLU \rightarrow Linear(4, 1)
$$

Os rótulos serão `0` para reprovado e `1` para aprovado. Durante o treinamento, a rede fará previsões, calculará a Binary Cross-Entropy, obterá os gradientes com backpropagation e atualizará seus parâmetros com um otimizador.

Neste primeiro exemplo, o objetivo principal não é construir uma arquitetura complexa, mas observar o ciclo completo de uma rede neural: entrada, processamento, previsão, cálculo da perda e atualização dos parâmetros.

In [ ]:
# Criar os dados de entrada e os rótulos.
# Definir a arquitetura da rede neural simples.
# Definir a função de custo e o otimizador.
# Implementar o ciclo de treinamento.
# Avaliar as previsões do modelo.


### Aprendizado Profundo

**Aprendizado profundo** (*deep learning*) é o uso de redes neurais com várias camadas de processamento. A palavra “profundo” se refere à quantidade de camadas entre a entrada e a saída, e não significa automaticamente que o modelo será melhor ou mais inteligente.

Em uma rede rasa, a entrada passa por poucas transformações. Em uma rede profunda, cada camada pode aprender uma representação intermediária diferente:

$$
h_1 = f_1(W_1x + b_1)
$$

$$
h_2 = f_2(W_2h_1 + b_2)
$$

$$
\hat{y} = g(W_3h_2 + b_3)
$$

As primeiras camadas podem aprender padrões mais simples. Em imagens, por exemplo, podem responder a bordas e formas básicas; em textos, podem capturar relações entre palavras. Camadas posteriores combinam essas representações para formar padrões mais abstratos e úteis para a tarefa.

A profundidade aumenta a capacidade de representação do modelo, mas também pode tornar o treinamento mais difícil. Redes profundas podem exigir mais dados, mais memória, mais tempo de treinamento e técnicas como inicialização adequada, normalização e regularização.

Também é importante distinguir capacidade de qualidade. Uma rede maior pode memorizar os dados de treinamento sem generalizar bem para exemplos novos. Por isso, o desempenho deve ser acompanhado em dados de validação e teste, e a arquitetura deve ser compatível com o problema.

Assim, uma rede neural simples é suficiente para entender o fluxo fundamental. O aprendizado profundo amplia esse mesmo fluxo com mais camadas, mais parâmetros e representações progressivamente mais elaboradas.

In [3]:
# Desenhar ou descrever uma arquitetura com várias camadas.
# Comparar uma rede rasa com uma rede profunda.
# Experimentar o efeito da quantidade de camadas e neurônios.

### Introdução a IA Moderna

Até aqui, vimos modelos que recebem dados, aprendem parâmetros e produzem previsões. A **Inteligência Artificial moderna** amplia essas ideias para modelos capazes de trabalhar com grandes volumes de dados e diferentes tipos de informação, como texto, imagem, áudio e vídeo.

É importante separar alguns conceitos relacionados:

- **Inteligência Artificial (IA):** área ampla que busca construir sistemas capazes de realizar tarefas associadas à inteligência, como reconhecer padrões, fazer previsões e tomar decisões.
- **Aprendizado de Máquina (Machine Learning):** abordagem em que o sistema aprende relações a partir de dados, em vez de receber todas as regras programadas manualmente.
- **Aprendizado Profundo (Deep Learning):** subárea do aprendizado de máquina baseada principalmente em redes neurais com muitas camadas.
- **IA Generativa:** modelos capazes de produzir novos conteúdos, como textos, imagens, códigos, áudios ou vídeos, a partir dos padrões aprendidos nos dados.

Uma forma de visualizar essa relação é:

$$
\text{IA} \supset \text{Machine Learning} \supset \text{Deep Learning}
$$

A IA moderna foi impulsionada por três fatores principais:

1. **Mais dados:** sistemas atuais podem aprender com grandes coleções de textos, imagens, sons e registros de uso.
2. **Mais capacidade computacional:** GPUs e outros aceleradores permitem realizar muitas operações matriciais em paralelo.
3. **Arquiteturas mais eficientes:** redes profundas, mecanismos de atenção e Transformers conseguem processar relações complexas entre os elementos dos dados.

Em muitos sistemas atuais, o modelo passa por duas etapas. Primeiro, ocorre o **pré-treinamento**, no qual ele aprende padrões gerais a partir de uma grande quantidade de dados. Depois, pode haver uma etapa de **ajuste** (*fine-tuning*), em que o modelo é adaptado para uma tarefa ou domínio específico.

O fato de um modelo aprender padrões não significa que ele compreenda o mundo da mesma forma que uma pessoa. Suas respostas dependem dos dados, da arquitetura, do objetivo de treinamento e do contexto fornecido. Por isso, sistemas de IA devem ser avaliados quanto à qualidade, generalização, segurança, privacidade e possíveis vieses.

Também é útil distinguir **modelo**, **dados** e **aplicação**. O modelo é o conjunto de parâmetros e operações aprendidos; os dados são os exemplos usados para treiná-lo ou consultá-lo; a aplicação é o sistema completo que combina o modelo com uma interface, regras, ferramentas e mecanismos de avaliação.

Nas próximas seções, veremos como os dados podem ser representados como embeddings e como a arquitetura Transformer usa atenção para modelar relações entre os elementos de uma sequência.

In [ ]:
# Comparar Inteligência Artificial, Machine Learning e Deep Learning.
# Identificar exemplos de IA preditiva e IA generativa.
# Relacionar dados, modelo, treinamento e aplicação.
# Discutir limitações, vieses, privacidade e avaliação de sistemas de IA.


### Embeddings

Computadores trabalham diretamente com números, mas textos, imagens e outros dados chegam inicialmente em formatos simbólicos. Uma rede neural não recebe a palavra `gato` como significado; ela recebe números. Por isso, precisamos de uma cadeia de representação: texto, tokens, índices e vetores.

**Tokenização** é o processo de dividir uma sequência em unidades menores chamadas tokens. Dependendo do tokenizador, um token pode ser uma palavra, parte de uma palavra, um sinal de pontuação, um espaço ou um símbolo especial. Por exemplo, a frase `gatos brincam` poderia ser dividida em tokens como `gatos` e `brincam`, enquanto uma tokenização por subpalavras poderia dividi-la em fragmentos reutilizáveis.

Depois da tokenização, cada token recebe um índice inteiro no vocabulário:

$$
[t_1,t_2,\ldots,t_n] \rightarrow [i_1,i_2,\ldots,i_n]
$$

O índice é apenas um identificador. O número 42 não é semanticamente maior ou menor que o número 17. Para que a rede consiga aprender relações, os índices são convertidos em vetores por uma tabela de embeddings.

Em vez de representar uma palavra apenas como um identificador, podemos associá-la a um vetor:

$$
\text{palavra} \rightarrow [v_1, v_2, \ldots, v_d]
$$

O número $d$ é a dimensão do embedding. Cada dimensão não precisa ter um significado humano isolado; o vetor completo é uma representação distribuída que o modelo usa para encontrar padrões.

Antes dos embeddings, uma alternativa comum era usar vetores **esparsos**, como one-hot ou bag-of-words. Em um vocabulário com $V$ tokens, um vetor one-hot possui dimensão $V$, com um único 1 e todos os demais valores iguais a zero:

$$
\text{gato} = [0,0,1,0,\ldots,0]
$$

Esse vetor é simples, mas não expressa que `gato` e `felino` podem estar relacionados. Além disso, vocabulários grandes produzem vetores enormes, muito esparsos e caros de armazenar e processar.

Vetores **densos** possuem muitos valores diferentes de zero e uma dimensão geralmente menor que o tamanho do vocabulário. Em vez de representar apenas a identidade de um token, um embedding denso pode aprender propriedades distribuídas a partir dos contextos:

$$
\text{gato} \rightarrow [0.21,-0.73,0.44,\ldots]
$$

A diferença é importante: a representação esparsa é principalmente uma identificação explícita; a representação densa é uma posição aprendida em um espaço contínuo. A tabela de embeddings pode ser vista como uma matriz $E \in \mathbb{R}^{V \times d}$, na qual cada linha corresponde ao vetor de um token.

Durante o treinamento, embeddings podem aprender relações úteis a partir do contexto em que os elementos aparecem. Palavras usadas em contextos semelhantes tendem a produzir vetores próximos em algum espaço de representação. Uma medida comum é a **similaridade de cosseno**:

$$
\operatorname{cos}(u,v) = \frac{u \cdot v}{\|u\|\|v\|}
$$

A similaridade vale 1 quando os vetores apontam para a mesma direção, 0 quando são ortogonais e pode se aproximar de -1 quando apontam em direções opostas. Como ela ignora parcialmente o tamanho do vetor e compara sua direção, é útil em busca semântica, recomendação e agrupamento. Ela não deve ser interpretada como uma medida perfeita de significado: o resultado depende dos dados, do modelo e do contexto de uso.

Existem dois usos importantes de embeddings. **Embeddings estáticos**, como os produzidos por Word2Vec, GloVe ou FastText, associam um vetor fixo a cada palavra ou unidade. Assim, a palavra `banco` tende a possuir o mesmo vetor em `banco financeiro` e `banco da praça`, embora os sentidos sejam diferentes.

Já os **embeddings contextuais** são calculados considerando a sequência em que o token aparece. A representação de `banco` pode mudar conforme as palavras ao redor. Modelos como BERT e Transformers produzem representações contextuais, permitindo distinguir usos diferentes da mesma forma textual.

Em modelos modernos, o embedding inicial do token é apenas o começo. Após passar por camadas de atenção e transformações, o modelo produz estados ocultos que incorporam informações do contexto. Por isso, é importante distinguir a linha da tabela de embeddings do vetor contextual produzido pela rede.

Embeddings não são dicionários perfeitos nem garantem compreensão. Eles são parâmetros aprendidos que representam regularidades dos dados. Por isso, podem refletir limitações, associações indesejadas ou vieses presentes no conjunto de treinamento.

Além de tokens, existem embeddings para outros tipos de objetos, como usuários, produtos, documentos, imagens e categorias. A ideia geral é a mesma: representar entidades em um espaço numérico no qual relações úteis possam ser aprendidas.

In [ ]:
# Criar tokens e associá-los a índices.
# Criar ou carregar uma tabela de embeddings.
# Comparar a distância ou a similaridade entre vetores.
# Observar as dimensões e o formato da representação.


### Introdução a Transformers e Attention

Uma sequência possui relações entre seus elementos. Em uma frase, por exemplo, uma palavra pode depender de outra que está distante. O mecanismo de **attention** permite que o modelo atribua pesos diferentes às partes da sequência ao produzir uma representação para cada posição. Em vez de tratar todos os tokens como igualmente relevantes, a rede aprende quais posições devem receber mais ou menos atenção.

A arquitetura **Transformer** foi apresentada no artigo *Attention Is All You Need*, de Ashish Vaswani e coautores, publicado em 2017. O trabalho mostrou que era possível construir uma arquitetura baseada principalmente em mecanismos de atenção, sem depender de redes recorrentes para processar uma sequência passo a passo.

A arquitetura **Transformer** foi apresentada no artigo *Attention Is All You Need*, de Ashish Vaswani e coautores, publicado em 2017. O trabalho mostrou que era possível construir uma arquitetura baseada principalmente em mecanismos de atenção, sem depender de redes recorrentes para processar uma sequência passo a passo. Isso favoreceu o processamento paralelo durante o treinamento e tornou mais simples modelar dependências distantes.

Para uma sequência com $n$ tokens, os embeddings formam uma matriz de entrada. Projeções lineares aprendidas transformam essa matriz em três matrizes:

$$
Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V
$$

Na atenção, cada elemento da sequência produz três representações:

- **Query (Q):** o que a posição atual está procurando;
- **Key (K):** as características usadas para comparar posições;
- **Value (V):** a informação que será combinada na saída.

A atenção escalada por produto interno pode ser representada por:

$$
\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Primeiro, $QK^T$ calcula compatibilidades entre todas as posições. O elemento na linha $i$ e coluna $j$ indica o quanto a posição $i$ deve considerar a posição $j$. Dividir por $\sqrt{d_k}$ ajuda a controlar a escala dos valores. A função softmax transforma cada linha em pesos que somam 1, e esses pesos são usados para combinar os valores em $V$.

Em um modelo autorregressivo, usamos uma máscara causal para impedir que uma posição consulte tokens futuros. Essa restrição é essencial quando o objetivo é prever o próximo token: durante o treinamento, a resposta correta não pode vazar para a entrada da posição atual.

O Transformer usa **multi-head attention**. Em vez de realizar uma única atenção, ele aprende várias cabeças que podem observar relações diferentes, como concordância, referência entre palavras ou proximidade semântica, e depois combina seus resultados. A arquitetura também inclui uma rede totalmente conectada por posição, conexões residuais e normalização.

Um bloco Transformer pode ser resumido como uma sequência de subcamadas: atenção, conexão residual e normalização; depois, uma rede feed-forward, outra conexão residual e outra normalização. A repetição desses blocos permite transformar progressivamente os embeddings iniciais em representações mais dependentes do contexto.

Como o Transformer não processa necessariamente os tokens em uma ordem recorrente, ele precisa receber informação sobre a posição de cada token. Para isso, são usados embeddings posicionais aprendidos ou codificações posicionais determinísticas. Assim, o modelo consegue distinguir sequências com os mesmos tokens em ordens diferentes.

Há diferentes formas de usar Transformers. Modelos com atenção bidirecional podem observar o contexto dos dois lados de um token e são úteis em tarefas de compreensão. Modelos autorregressivos usam o contexto anterior para prever o próximo token e são adequados para geração. Essa diferença está relacionada ao objetivo de treinamento e ao tipo de tarefa.

A atenção tem custo que cresce aproximadamente com o quadrado do comprimento da sequência, pois compara cada posição com muitas outras. Por isso, contextos longos exigem memória e computação consideráveis, e existem pesquisas e arquiteturas alternativas para reduzir esse custo.

In [ ]:
# Representar uma sequência como tokens e embeddings.
# Calcular uma matriz simples de similaridade entre posições.
# Aplicar softmax para obter pesos de atenção.
# Combinar os valores usando esses pesos.


### LLMs e ChatGPT

**LLM** significa *Large Language Model*, ou grande modelo de linguagem. É um modelo treinado para aprender padrões estatísticos de uma linguagem e, em muitos casos, prever o próximo token de uma sequência.

Em uma formulação autorregressiva, dado um contexto de tokens $x_1, x_2, \ldots, x_t$, o modelo estima uma distribuição para o próximo token:

$$
P(x_{t+1}\mid x_1, x_2, \ldots, x_t)
$$

Durante a geração, um token é escolhido, anexado ao contexto e usado para prever o próximo. Esse processo se repete até que o modelo atinja um token de parada ou o limite definido. A saída é produzida passo a passo, mesmo que o usuário perceba a resposta como um texto único.

O treinamento de um LLM costuma envolver uma grande quantidade de dados e parâmetros. No pré-treinamento, o modelo aprende a prever tokens e ajusta seus parâmetros para reduzir a função de custo. Depois, ele pode passar por etapas adicionais para seguir instruções, adaptar-se a tarefas específicas e melhorar sua utilidade em conversas.

O **ChatGPT** é uma aplicação conversacional baseada em modelos de linguagem da família GPT (*Generative Pre-trained Transformer*). O modelo recebe uma sequência de mensagens como contexto e gera uma continuação token por token. A aplicação ao redor do modelo também pode incluir instruções, histórico da conversa, filtros, ferramentas e outras regras de produto.

É importante distinguir o modelo da interface. Um LLM é o componente que calcula distribuições e gera tokens; um sistema como um chatbot combina esse modelo com uma interface e mecanismos adicionais para organizar a interação.

Um LLM não consulta automaticamente uma fonte de verdade universal. Ele gera respostas com base nos padrões aprendidos e no contexto disponível. Por isso, pode produzir informações incorretas, inventar referências ou apresentar uma resposta plausível com excesso de confiança. A verificação é especialmente importante em temas técnicos, científicos, jurídicos, médicos e em qualquer situação de alto impacto.

A avaliação de um sistema conversacional deve considerar mais do que fluência. Também precisamos observar factualidade, coerência, segurança, privacidade, robustez a instruções inadequadas, vieses e capacidade de reconhecer incerteza.

Nesta aula, o objetivo não é tratar o ChatGPT como uma caixa-preta mágica, mas conectar seus componentes aos conceitos estudados: dados são tokenizados, tokens são convertidos em embeddings, Transformers calculam relações por atenção e o modelo aprende parâmetros por meio de custo, backpropagation e otimização.

In [ ]:
# Tokenizar uma frase e observar os tokens gerados.
# Representar a previsão do próximo token como uma distribuição.
# Comparar diferentes estratégias de geração.
# Discutir verificação de respostas, limitações e uso responsável.
